# Verdict RAG pipeline

This notebook documents and evaluates the grounded competitive-programming tutor. It follows the graduation brief: inspect a corpus, chunk it, embed into a persistent vector store, retrieve evidence, generate with Ollama, cite sources, and evaluate at least ten questions.

The production UI is the stripped Verdict mirror. The notebook is the reproducible RAG evidence and evaluation surface.

In [ ]:
from pathlib import Path
import json, re, sys
ROOT = Path.cwd()
if (ROOT / 'backend').exists():
    sys.path.insert(0, str(ROOT / 'backend'))
else:
    ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT / 'backend'))
CORPUS = ROOT / 'backend' / 'data' / 'knowledge'
EVAL = ROOT / 'backend' / 'evaluation' / 'questions.json'
print('Project root:', ROOT)
print('Corpus files:', len(list(CORPUS.glob('*.md'))))

## 1. Load and inspect the corpus

The curated corpus contains practical notes for binary search, two pointers, prefix sums, graph traversal, and dynamic programming. Each file includes the invariant, implementation pattern, complexity, and pitfalls. These focused documents reduce retrieval noise compared with indexing the entire source tree.

In [ ]:
documents = []
for path in sorted(CORPUS.glob('*.md')):
    text = path.read_text(encoding='utf-8')
    documents.append({'path': str(path), 'title': path.stem, 'text': text})
for doc in documents:
    print(f"{doc['title']}: {len(doc['text'])} characters")

## 2. Chunking

Chunks are 1,200 characters with a 160-character overlap (the same settings used by `backend/scripts/ingest.py`). The overlap preserves definitions and complexity notes that cross a heading boundary. Hash-based IDs make ingestion repeatable and safe to rerun.

In [ ]:
def chunk_text(text, size=1200, overlap=160):
    clean = re.sub(r'\s+', ' ', text).strip()
    step = max(1, size - overlap)
    return [clean[i:i+size] for i in range(0, len(clean), step) if clean[i:i+size].strip()]
chunks = []
for doc in documents:
    for index, chunk in enumerate(chunk_text(doc['text'])):
        chunks.append({'id': f"{doc['title']}::{index}", 'title': doc['title'], 'chunk': index, 'text': chunk})
print('Total chunks:', len(chunks))
print(chunks[0]['text'][:300])

## 3. Persist embeddings in Chroma

The application uses `sentence-transformers/all-MiniLM-L6-v2` through Chroma's embedding function. The persistent directory is `data/chroma`; it is ignored by Git because model and index files are generated artifacts.

In [ ]:
from app.core.config import Settings
from app.services.retrieval import Retriever
settings = Settings()
retriever = Retriever(settings.chroma_persist_dir, settings.chroma_collection, settings.embedding_model)
indexed = retriever.add([x['text'] for x in chunks], [x['id'] for x in chunks], [{
    'source_id': x['id'], 'title': x['title'], 'source': x['title'] + '.md', 'chunk': x['chunk']
} for x in chunks])
print('Indexed:', indexed, 'stored:', retriever.count())

## 4. Retrieval checks

A useful retrieval result should name a relevant source and expose enough context for a grounded answer. The production API returns these same records as citations.

In [ ]:
def show_retrieval(question, k=3):
    results = retriever.search(question, k)
    for rank, result in enumerate(results, 1):
        print(f"{rank}. {result['metadata'].get('title')} (score={result.get('score')})")
        print(result['document'][:220].replace('\n', ' '), '\n')
    return results
show_retrieval('How do I prove a binary search loop is correct?')

## 5. Ten-plus question evaluation

The hand-labeled set includes in-domain algorithm questions and out-of-domain questions. `Hit@k` measures whether the expected topic appears in the retrieved source title. Out-of-domain questions are expected to return no labeled topic and are used to inspect abstention behavior.

In [ ]:
questions = json.loads(EVAL.read_text(encoding='utf-8'))
rows = []
for item in questions:
    results = retriever.search(item['question'], 3)
    titles = [r['metadata'].get('title', '').lower().replace('_', ' ') for r in results]
    expected = item.get('expected_source')
    hit = bool(expected and expected.replace('_', ' ') in ' '.join(titles))
    rows.append({**item, 'hit_at_3': hit, 'retrieved': titles})
labeled = [r for r in rows if r.get('expected_source')]
print(f"Hit@3: {sum(r['hit_at_3'] for r in labeled)}/{len(labeled)} = {sum(r['hit_at_3'] for r in labeled)/len(labeled):.1%}")
for row in rows:
    print(row['id'], 'hit' if row['hit_at_3'] else 'miss', row['retrieved'])

## 6. Grounded generation and citation checks

Generation is deliberately constrained to retrieved context. If Chroma returns no context, the Ollama prompt tells the model to state that limitation rather than inventing an answer or citation. The API response exposes `grounded`, `retrieved_context`, `citations`, model name, and latency for inspection.

In [ ]:
import asyncio
from app.services.generation import Generator
generator = Generator(settings)
context = retriever.search('When should I use prefix sums?', 3)
answer = asyncio.run(generator.generate('When should I use prefix sums?', context, 'teach'))
print(answer)
print('Citation markers present:', bool(re.search(r'\[\d+\]', answer)))

## 7. Failure analysis

Common failure modes are visible and testable: no index, no Ollama process, an out-of-domain question, or an ambiguous problem statement. The service remains healthy when optional dependencies are unavailable and returns a clear Ollama-unavailable answer instead of fabricating content.

In [ ]:
ood = next((q for q in questions if not q.get('expected_source')), {'question': 'What is the weather today?'})
ood_results = retriever.search(ood['question'], 3)
print('OOD question:', ood['question'])
print('Retrieved titles:', [r['metadata'].get('title') for r in ood_results])
print('Production behavior:', 'answer with limitation when context is empty; never invent citations')

### Reproduce from a terminal

```bash
cd backend
python -m scripts.ingest --input data/knowledge
python -m scripts.evaluate_retrieval --persist-dir data/chroma
pytest -q
```

Start Ollama with `ollama serve` and the configured model before testing generated answers. The FastAPI service is launched with `uvicorn app.main:app --reload --port 8000`.